# Capstone: Security ML Pipeline

## What This Is
This capstone assembles a complete security ML pipeline:

1. **Ingest**: Network flows + endpoint telemetry + logs
2. **Detect**: IDS (network), UEBA (user), malware classifier (endpoint)
3. **Triage**: Multi-signal scoring, severity assignment
4. **Hunt**: Unsupervised outlier detection for uncategorized threats
5. **Report**: Structured incident summary with IOCs and recommendations

This mirrors a real SOC (Security Operations Center) workflow, with ML at each stage.

In [ ]:
import numpy as np
import json
from typing import Dict, List, Optional

np.random.seed(42)

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

class SecurityMLPipeline:
    """End-to-end security ML pipeline."""
    
    def __init__(self):
        self.network_model = None     # IDS: w, b
        self.malware_model = None     # Malware classifier: w, b
        self.user_baseline = {}       # UEBA: per-user baseline stats
        self.anomaly_threshold = 0.5
        self.trained = False
    
    def train(self, network_X, network_y, malware_X, malware_y, user_logs):
        # Train network IDS
        mu, s = network_X.mean(0), network_X.std(0) + 1
        Xn = (network_X - mu) / s
        w = np.zeros(Xn.shape[1]); b = 0.0
        for _ in range(200):
            e = sigmoid(Xn @ w + b) - network_y
            w -= 0.05 * (Xn.T @ e) / len(network_y); b -= 0.05 * e.mean()
        self.network_model = (w, b, mu, s)
        
        # Train malware classifier
        mu2, s2 = malware_X.mean(0), malware_X.std(0) + 1
        Xm = (malware_X - mu2) / s2
        w2 = np.zeros(Xm.shape[1]); b2 = 0.0
        for _ in range(200):
            e2 = sigmoid(Xm @ w2 + b2) - malware_y
            w2 -= 0.05 * (Xm.T @ e2) / len(malware_y); b2 -= 0.05 * e2.mean()
        self.malware_model = (w2, b2, mu2, s2)
        
        self.trained = True
        print('Pipeline trained: IDS + Malware classifier')
    
    def analyze_network(self, flow: np.ndarray) -> Dict:
        w, b, mu, s = self.network_model
        score = float(sigmoid((flow - mu) / s @ w + b))
        return {'type': 'network', 'score': score, 
                'verdict': 'MALICIOUS' if score > 0.5 else 'BENIGN'}
    
    def analyze_file(self, pe_features: np.ndarray) -> Dict:
        w, b, mu, s = self.malware_model
        score = float(sigmoid((pe_features - mu) / s @ w + b))
        return {'type': 'file', 'score': score,
                'verdict': 'MALWARE' if score > 0.5 else 'CLEAN'}
    
    def generate_incident_report(self, findings: List[Dict]) -> str:
        alerts = [f for f in findings if f.get('verdict') in ['MALICIOUS', 'MALWARE']]
        severity = 'CRITICAL' if len(alerts) > 1 else ('HIGH' if alerts else 'LOW')
        
        report = {
            'severity': severity,
            'total_analyzed': len(findings),
            'alerts_generated': len(alerts),
            'network_alerts': sum(1 for a in alerts if a['type'] == 'network'),
            'malware_alerts': sum(1 for a in alerts if a['type'] == 'file'),
            'max_score': max((f['score'] for f in findings), default=0),
            'recommendation': 'ISOLATE_HOST' if severity == 'CRITICAL' else 'INVESTIGATE',
        }
        return json.dumps(report, indent=2)

pipeline = SecurityMLPipeline()
print('Security ML Pipeline initialized')


In [ ]:
# Train and run the pipeline

# Generate training data
def gen_net_flow(is_attack, n):
    if is_attack:
        return np.column_stack([
            np.random.exponential(0.2, n), np.ones(n)*6,
            np.random.poisson(500,n), np.random.poisson(2,n),
            np.random.poisson(500,n)*60, np.random.poisson(2,n)*50,
            np.random.exponential(50000,n), np.random.exponential(1000,n),
        ])
    else:
        return np.column_stack([
            np.random.exponential(2.0,n), np.random.choice([6,17],n),
            np.random.poisson(15,n), np.random.poisson(12,n),
            np.random.poisson(15,n)*500, np.random.poisson(12,n)*500,
            np.random.exponential(5000,n), np.random.exponential(30,n),
        ])

def gen_pe(is_mal, n):
    if is_mal:
        return np.column_stack([
            np.random.randint(20000,500000,n), np.random.randint(3,8,n),
            np.random.uniform(6.5,7.9,n), np.random.uniform(5.0,7.5,n),
            np.random.randint(5,30,n), np.random.randint(2,8,n),
            np.random.binomial(1,0.7,n), np.random.binomial(1,0.3,n),
            np.random.randint(50,500,n), np.random.randint(1,10,n),
            np.random.randint(0,5,n), np.random.randint(0,3,n),
            np.random.binomial(1,0.4,n)
        ], dtype=float)
    else:
        return np.column_stack([
            np.random.randint(50000,5000000,n), np.random.randint(4,12,n),
            np.random.uniform(5.0,6.5,n), np.random.uniform(2.0,5.0,n),
            np.random.randint(30,200,n), np.random.randint(0,2,n),
            np.random.binomial(1,0.05,n), np.random.binomial(1,0.9,n),
            np.random.randint(200,5000,n), np.random.randint(0,2,n),
            np.random.randint(0,3,n), np.random.randint(0,50,n),
            np.random.binomial(1,0.02,n)
        ], dtype=float)

net_X = np.vstack([gen_net_flow(False, 600), gen_net_flow(True, 200)])
net_y = np.array([0]*600 + [1]*200)
mal_X = np.vstack([gen_pe(False, 600), gen_pe(True, 400)])
mal_y = np.array([0]*600 + [1]*400)

pipeline.train(net_X, net_y, mal_X, mal_y, {})

# Simulate an investigation scenario
print('\nAnalyzing incident...')
findings = []
# Normal flows
for x in gen_net_flow(False, 5):
    findings.append(pipeline.analyze_network(x))
# Attack flows (suspicious)
for x in gen_net_flow(True, 3):
    findings.append(pipeline.analyze_network(x))
# Malware files
for x in gen_pe(True, 2):
    findings.append(pipeline.analyze_file(x))

print(pipeline.generate_incident_report(findings))
